<a href="https://colab.research.google.com/github/redinbluesky/handson-llm/blob/main/02_토큰과_임베딩.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#  목차
* [Chapter 2 서론](#chapter2)
* [Chapter 2-1 LLM 토큰화](#chapter2-1)
    * [Chapter 2-1-1 토크나이져가 언어 모델의 입력을 준비하는 방법](#chapter2-1-1)
    * [Chapter 2-1-2 LLM 다운로드하고 실행하기](#chapter2-1-2)
    * [Chapter 2-1-3 토크나이저가 텍스트를 분할하는 방법](#chapter2-1-3)
    * [Chapter 2-1-4 단어 토큰, 부분단어 토큰, 문자 토큰, 바이트 토큰](#chapter2-1-4)
    * [Chapter 2-1-5 훈련된 LLM 토크나이져 비교하기](#chapter2-1-5)

## Chapter 2 서론 <a class="anchor" id="chapter2"></a>
1. 토큰과 임베딩은 대규모 언어 모델을 사용하느 데 중심이 되는 두가지 개념이다.
    - 토큰과 임베딩을 이해하지 못하면 대규모 언어 모델이 어떻게 작동하는지 이해하기 어렵다.
    - 언어 모델은 텍스트를 토큰이라는 작은 단위로 나우어 처리한다.
    - 언어 모델은 언어에 대한 계산을 수행하기 위해 토큰을 임베딩이라는 벡터로 변환한다.

        ![토큰과 임베딩](./image/02_token_embedding.png)

## Chapter 2-1 LLM 토큰화 <a class="anchor" id="chapter2-1"></a>
1. 현재 대부분의 사용자의 언어모델을 사용하는 방법은 웹 애플리케이션으로 제공되는 채팅 인터페이스를 사용하는 것이다.
    - 예시) ChatGPT, Bing Chat, Google Bard 등

2. 이러한 모델은 한 번에 모든 출력을 만들지는 않는다.
    - 모델은 텍스트를 토큰이라는 작은 단위로 나누어 처리한다.
    - 모델은 토큰을 하나씩 생성하여 출력을 만든다.

3. 토큰은 모델의 입력에도 사용된다.
    - 모델은 입력 텍스트를 토큰으로 나누어 처리한다.
    - 모델은 입력 토큰을 임베딩이라는 벡터로 변환하여 계산을 수행한다.

### Chapter 2-1-1 토크나이져가 언어 모델의 입력을 준비하는 방법 <a class="anchor" id="chapter2-1-1"></a>
1. 입력을 프롬프트를 모델에 전달하기 전에 토크나이져를 통과시켜 더 작은 단위로 쪼개야한다.
    - 아래의 그림과 같이 토크나이져가 텍스트를 단어나 부분단어로 나눈다.
    
        ![토크나이져 예시](./image/02_tokenizer_example2.png)

### Chapter 2-1-2 LLM 다운로드하고 실행하기 <a class="anchor" id="chapter2-1-2"></a>


In [1]:
# 깃허브에서 위젯 상태 오류를 피하기 위해 진행 표시줄을 나타내지 않도록 설정합니다.
from transformers.utils import logging

logging.disable_progress_bar()

In [29]:
# 트랜스포머스에서 "microsoft/Phi-3-mini-4k-instruct" 모델과 토크나이져 로드
from transformers import AutoModelForCausalLM, AutoTokenizer

# 모델과 토크나이저를 로드합니다.
model = AutoModelForCausalLM.from_pretrained(
    "microsoft/Phi-3-mini-4k-instruct",
    device_map="cuda",
    dtype="auto"
)
tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-3-mini-4k-instruct")

1. 프롬프트를 준비하고 토큰으로 분리한다.
    - 토큰을 모델에 전달하여 출력을 생성한다.
    - 모델에게 새로운 토큰 20개만 생성하도록 요청한다.

In [30]:
# 프롬프트 생성
#   - 사라에게 정원 가꾸기 사고에 대해 사과하는 이메일 작성
#   - <|assistant|> 토큰은 모델이 응답을 생성하기 시작하는 위치를 나타냅니다.
prompt = "Write an email apologizing to Sarah for the tragic gardening mishap. Explain how it happened.<|assistant|>"

# 입력 프롬프트를 토큰으로 변환
#   - 토크나이저는 텍스트를 모델이 이해할 수 있는 숫자 시퀀스로 변환합니다.
#   - return_tensors="pt"는 토큰이 PyTorch 텐서로 반환되도록 지정합니다.
#   - .input_ids는 토큰의 숫자 ID 시퀀스를 추출합니다.
input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to("cuda")
input_ids

tensor([[14350,   385,  4876, 27746,  5281,   304, 19235,   363,   278, 25305,
           293, 16423,   292,   286,   728,   481, 29889, 12027,  7420,   920,
           372,  9559, 29889, 32001]], device='cuda:0')

In [31]:
for id in input_ids[0]:
   print(tokenizer.decode(id))

Write
an
email
apolog
izing
to
Sarah
for
the
trag
ic
garden
ing
m
ish
ap
.
Exp
lain
how
it
happened
.
<|assistant|>


In [32]:
print(tokenizer.decode(3323))
print(tokenizer.decode(622))
print(tokenizer.decode([3323, 622]))
print(tokenizer.decode(29901))

Sub
ject
Subject
:


2. 모델이 직접 텍스트 프롬프트를 입력 받지 않는다.
    - 토크나이저가 입력 프롬프트를 처리하여 input_ids라는 토큰의 시퀀스로 변환한다.

3. input_ids에는 모델이 이해할 수 있는 일련의 정수가 포함되어있다.
    - 각 정수는 어휘사전에 있는 틀정 토큰에 매핑된다.

4. 그림으로 표현하면 아래와 같이 입력이 토큰으로 나누어져 모델에 전달된다.

    ![토큰화 과정](./image/02_tokenization_process.png)

5. 토크나이져로 분리된 토큰은 아래의 주의사항이 있다.
    - 일부 토큰은 완전한 단어이다.
    - 일부 토큰은 부분단어이다.
    - 구두점 문자도 하나의 토큰이다.
    - 공백 문자는 토큰으로 취급하지 않지만 토큰의 앞이 공백인경우 "_"를 문자앞에 추가한다.

### Chapter 2-1-3 토크나이저가 텍스트를 분할하는 방법 <a class="anchor" id="chapter2-1-3"></a>
1. 모델 설계 시에 모델 작성자가 토큰호 방법을 선택한다.
    - GPT는 BPE(Byte Pair Encoding)라는 방법을 사용한다.
    - BERT는 WordPiece라는 방법을 사용한다.

2. 토큰화 방법을 선택 후 어휘사전 크기와 특수 토큰 같은 세부 사항을 결정한다.
    - 어휘사전 크기는 모델이 사용할 수 있는 고유 토큰의 수를 결정한다.
    - 특수 토큰은 모델이 입력과 출력을 구분하는 데 사용된다.
    - 예시) [CLS], [SEP], [PAD] 등

3. 토크나이저는 특정 데이터셋에서 훈련하여 해당 데이터셋을 가장 잘 표현할 수 있는 토큰화 방법을 학습한다.
    - 훈련된 토크나이저는 모델이 훈련된 데이터와 유사한 텍스트를 처리하는 데 최적화되어 있다.
    - 영어 데이터셋에서 훈련된 토크나이져와 다국어 데이터셋에서 훈련된 토크나이져는 동일한 방법과 파라미터를 사용하더라도 다른 토큰화 결과를 낼 수 있다.

4. 토크나이져는 입력 텍스트뿐 아니라 모델이 생성하는 출력 텍스트도 토큰화한다.
    - 모델이 생성하는 출력 텍스트도 토크나이져가 처리하여 토큰으로 나누어진다.
    - 모델이 생성하는 출력 텍스트는 모델이 훈련된 데이터와 유사한 텍스트를 생성하는 경향이 있다.

### Chapter 2-1-4 단어 토큰, 부분단어 토큰, 문자 토큰, 바이트 토큰 <a class="anchor" id="chapter2-1-4"></a>
1. 단어 토큰
    - 단어 토큰은 완전한 단어를 나타내는 토큰이다.
        - 예시) "hello", "world", "token" 등
    - 토크나이저가 훈련 후에 데이터셋에 새롭게 추가된 단어를 처리할 수 없다.
        - 최대한 많은 단어를 어휘사전에 포함시키려고 하지만 새로운 단어가 계속해서 등장하기 때문에 모든 단어를 포함시키는 것은 불가능하다.
    
2. 부분단어 토큰
    - 부분단어 토큰은 단어의 일부를 나타내는 토큰이다
        - 예시) "un", "##happy", "##ness" 등
    - 부분단어 토큰은 어휘사전에 없는 단어를 처리할 수 있도록 도와준다.
        - 예시) "unhappy"라는 단어가 어휘사전에 없지만 "un"과 "##happy"라는 부분단어 토큰이 어휘사전에 있다면 "unhappy"라는 단어를 "un"과 "##happy"로 나누어 처리할 수 있다.

3. 문자 토큰
    - 문자 토큰은 단어의 개별 문자를 나타내는 토큰이다
        - 예시) "h", "e", "l", "o" 등
    - 문자 토큰은 어휘사전에 없는 단어를 처리할 수 있도록 도와준다.
        - 예시) "hello"라는 단어가 어휘사전에 없지만 "h", "e", "l", "o"라는 문자 토큰이 어휘사전에 있다면 "hello"라는 단어를 "h", "e", "l", "o"로 나누어 처리할 수 있다.
    - hello를 조합하는 정보까지 모델링해야하는 부담이 있다.
    
4. 바이트 토큰
    - 유니코드 문자를 표현하는 바이트 시퀀스를 나타내는 토큰이다
        - 예시) "h"는 0x68, "e"는 0x65, "l"는 0x6C, "o"는 0x6F 등

5. 토큰화 방법을 이미지로 표현하면 아래와 같다.

    ![토큰화 방법](./image/02_tokenization_methods.png)

6. 일부 부분단어 토크나이져는 어휘사전에 바이트를 토큰으로 포함하기도 한다.
    - 표현할 수 없는 문자가 있을 때 바이트 토큰을 사용하여 처리할 수 있다.

### Chapter 2-1-5 훈련된 LLM 토크나이져 비교하기 <a class="anchor" id="chapter2-1-5"></a>

In [33]:
from transformers import AutoModelForCausalLM, AutoTokenizer

colors_list = [
    '102;194;165', '252;141;98', '141;160;203',
    '231;138;195', '166;216;84', '255;217;47'
]

def show_tokens(sentence, tokenizer_name):
    tokenizer = AutoTokenizer.from_pretrained(tokenizer_name)
    token_ids = tokenizer(sentence).input_ids
    for idx, t in enumerate(token_ids):
        # 텍스트를 인코딩한 후 다시 디코딩했을 때 원본 텍스트와 동일해지려면
        # clean_up_tokenization_spaces를 False로 지정해야 합니다.
        # 현재 이 매개변수의 기본값은 None(True에 해당)이며
        # transformers 4.45에서 True로 바뀔 예정입니다.
        # https://github.com/huggingface/transformers/issues/31884
        print(
            f'\x1b[0;30;48;2;{colors_list[idx % len(colors_list)]}m' +
            tokenizer.decode(t, clean_up_tokenization_spaces=False) +
            '\x1b[0m',
            end=' '
        )

In [34]:
text = """
English and CAPITALIZATION
🎵 鸟
show_tokens False None elif == >= else: two tabs:"		" four spaces:"    "
12.0*50=600
"""

In [36]:
show_tokens(text, "bert-base-uncased")

[CLS] english and capital ##ization [UNK] [UNK] show _ token ##s false none eli ##f = = > = else : two tab ##s : " " four spaces : " " 12 . 0 * 50 = 600 [SEP] 

In [ ]:
11